# QCC Transformer：真实 1M Qwen / CUDA 验证

选择 Colab GPU 后按顺序运行。这个 notebook 只运行真实 checkpoint 配置检查、仓库 CUDA 测试和 HF retrofit 接口验证；完整的 1M Full-KV/QCC 配对质量与 serving 数据需要能容纳该模型和 Full-KV 的运行时。

In [ ]:
%cd /content
!git clone --depth 1 --branch codex/qcc-exact-tier-20260903 https://github.com/Marchematics/qcc-transformer.git /content/qcc-transformer
%cd /content/qcc-transformer
%pip install -e ".[hf]"
%pip install -U triton sentencepiece accelerate

In [ ]:
import os
import torch
from transformers import AutoConfig, AutoTokenizer
os.chdir("/content/qcc-transformer")
model_id = "Qwen/Qwen2.5-7B-Instruct-1M"
config = AutoConfig.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print({
    "model": model_id,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "max_position_embeddings": getattr(config, "max_position_embeddings", None),
    "layers": getattr(config, "num_hidden_layers", None),
    "query_heads": getattr(config, "num_attention_heads", None),
    "kv_heads": getattr(config, "num_key_value_heads", None),
    "vocab_size": len(tokenizer),
})

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_archive.py", "tests/test_vllm.py", "tests/test_retrofit.py"], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f"pytest failed: {result.returncode}")

In [ ]:
from accelerate import init_empty_weights
from transformers import AutoModelForCausalLM
from qcc_transformer import HFQCCAttention, patch_hf_model
with init_empty_weights():
    model = AutoModelForCausalLM.from_config(config)
paths = patch_hf_model(model, kv_head_policy="repeat", use_triton=False, max_position_embeddings=1_010_000)
wrappers = [module for module in model.modules() if isinstance(module, HFQCCAttention)]
total = sum(parameter.numel() for parameter in model.parameters())
adapter = sum(parameter.numel() for wrapper in wrappers for parameter in wrapper.qcc.archive.parameters())
adapter += sum(parameter.numel() for wrapper in wrappers for parameter in wrapper.qcc.gate.parameters())
print({"patched_layers": len(paths), "adapter_parameters": adapter, "backbone_parameters": total, "adapter_fraction": adapter / max(total, 1), "gqa": (config.num_attention_heads, config.num_key_value_heads)})

In [ ]:
from qcc_transformer.vllm_stock import QCCPackedStateLayout, QCCStockVLLMConfig
from qcc_transformer.stock_runtime import packed_ratio_vs_full_kv
stock_config = QCCStockVLLMConfig(
    num_heads=int(config.num_attention_heads),
    head_dim=int(config.hidden_size // config.num_attention_heads),
    num_codes=64,
    num_scales=4,
    exact_num_sets=128,
    exact_ways=4,
    max_position_embeddings=int(config.max_position_embeddings),
    local_element_bytes=2,
)
layout = QCCPackedStateLayout(stock_config)
rows = []
for context_tokens in (128000, 256000, 512000, 1000000):
    ratio = packed_ratio_vs_full_kv(
        layout, context_tokens=context_tokens, num_kv_heads=int(config.num_key_value_heads)
    )
    rows.append({"context_tokens": context_tokens, "state_bytes_per_layer_request": layout.mutable_state_bytes(), "full_kv_ratio": ratio, "state_reduction": 1.0 - ratio})
print({"model": model_id, "head_dim": stock_config.head_dim, "rows": rows})
print("These are fixed-layout byte counts, not measured serving or quality results.")

In [ ]:
import importlib.util
print({"bitsandbytes_installed": importlib.util.find_spec("bitsandbytes") is not None, "quantized_command": "pip install -e .[hf-quant] then add --load-in-4bit to a real HF benchmark"})

In [ ]:
# Optional real-model latency sample on a T4; use a larger GPU for target-context runs.
import subprocess, sys
command = [
    sys.executable, "benchmarks/benchmark_hf_latency.py",
    "--model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--prompt", "A real pretrained-model latency test. " * 64,
    "--device", "cuda", "--dtype", "float16",
    "--window-size", "64", "--num-codes", "32",
    "--num-decode-steps", "8", "--repeats", "3",
    "--warmup", "1", "--kv-head-policy", "repeat",
]
run = subprocess.run(command, text=True, capture_output=True)
print(run.stdout)
print(run.stderr)
if run.returncode:
    raise RuntimeError(f"real latency run failed: {run.returncode}")

## 运行真实长上下文实验

用 `benchmarks/benchmark_hf_retrieval_1m.py`、RULER、LongBench 和 PG-19 的现有入口加载同一个真实 checkpoint 与 adapter。Qwen 官方 1M checkpoint 的 Full-KV 对照需要远大于 T4 的显存；在没有足够显存前，不把 QCC-only 长流或较短模型结果写成 1M 配对质量。